In [1]:
!pip install -q sentence-transformers

In [ ]:
pip install -U bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00


In [ ]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.7 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
from google.colab import drive
import numpy as np
from collections import Counter
from typing import Optional
import torch
from datasets import load_dataset
from src.data.schema import QASample, SamplerConfig
import random
import time
import math

from sentence_transformers import SentenceTransformer

from dataclasses import dataclass, field

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
assert DEVICE == "cuda", "Switch to GPU runtime: Runtime > Change runtime type > T4 GPU"

Device: cuda


In [ ]:
from model import load_sentence_encoder, load_model, generate_response, generate_batch
from scoring import clean_response, compute_stability, is_uncertainty_response, plateau_stop, responses_are_substantive
from data import load_triviaqa, load_mmlu, load_webquestions
from evaluate import evaluate, run_full_evaluation, print_comparison_table
from calibration import compute_qhat, check_coverage, run_calibration
from sampler import adaptive_sample, check_coverage, build_prediction_set
from helpers import _assign_split, _print_split_counts, filter_split, clear_embed_cache, embed_cached
from scoring import clean_response, compute_stability, plateau_stop, is_uncertainty_response, responses_are_substantive, token_f1
from config import QASample, SamplerConfig, UNCERTAINTY_TOKENS, COVERAGE_SIM_THRESHOLD, MMLU_SUBJECTS

In [ ]:
def test_semantic_coverage():
    print("Testing semantic check_coverage...\n")

    assert check_coverage([], ["shakespeare"]) == True
    print("  PASS  abstention → covered")

    assert check_coverage(["eastern"], ["north american eastern time zone"]) == True
    print("  PASS  'eastern' covers 'north american eastern time zone'")

    assert check_coverage(["durbin"], ["dick durbin"]) == True
    print("  PASS  'durbin' covers 'dick durbin'")

    assert check_coverage(["shakespeare"], ["william shakespeare"]) == True
    print("  PASS  'shakespeare' covers 'william shakespeare'")

    assert check_coverage(["napoleon"], ["shakespeare"]) == False
    print("  PASS  'napoleon' does not cover 'shakespeare'")

    assert check_coverage(["kansas"], ["united states of america"]) == False
    print("  PASS  'kansas' does not cover 'united states of america' (genuinely wrong)")

    print("\nAll semantic coverage tests passed.\n")

test_semantic_coverage()

Testing semantic check_coverage...

  PASS  abstention → covered
  PASS  'eastern' covers 'north american eastern time zone'
  PASS  'durbin' covers 'dick durbin'
  PASS  'shakespeare' covers 'william shakespeare'
  PASS  'napoleon' does not cover 'shakespeare'
  PASS  'kansas' does not cover 'united states of america' (genuinely wrong)

All semantic coverage tests passed.



In [ ]:
# def is_uncertainty_response(response: str) -> bool:
#     """Return True if response expresses ignorance rather than a fact."""
#     r = response.lower().strip()
#     return any(token in r for token in UNCERTAINTY_TOKENS)

# def responses_are_substantive(cleaned_responses: list,
#                                uncertainty_threshold: float = 0.6) -> bool:
#     """
#     Return True if the majority of responses are substantive answers.

#     If more than uncertainty_threshold fraction are uncertainty expressions,
#     treat the response set as non-substantive → force abstention.
#     """
#     if not cleaned_responses:
#         return False

#     n_uncertain = sum(1 for r in cleaned_responses
#                       if is_uncertainty_response(r))
#     return (n_uncertain / len(cleaned_responses)) < uncertainty_threshold

In [ ]:
def run_all_unit_tests():

    print("=" * 55)
    print("UNIT TESTS")
    print("=" * 55)

    # --- compute_stability ---
    print("\n[compute_stability]")

    result = compute_stability(["shakespeare"] * 4)
    assert result > 0.99, f"Expected ~1.0, got {result}"
    print(f"  PASS  identical responses → {result:.4f}")
    clear_embed_cache()

    result = compute_stability(["william shakespeare", "shakespeare",
                                 "william shakespeare", "shakespeare"])
    assert result > 0.85, f"Expected > 0.85, got {result}"
    print(f"  PASS  name variants → {result:.4f}")
    clear_embed_cache()

    result = compute_stability(["shakespeare", "napoleon", "einstein"])
    assert result < 0.75, f"Expected < 0.75, got {result}"
    print(f"  PASS  different people → {result:.4f}")
    clear_embed_cache()

    # --- plateau_stop ---
    print("\n[plateau_stop]")

    assert plateau_stop([0.60, 0.74, 0.75, 0.75], 0.03) == True
    print("  PASS  flat history → stop")

    assert plateau_stop([0.40, 0.55, 0.68, 0.75], 0.03) == False
    print("  PASS  rising history → no stop")

    assert plateau_stop([0.60, 0.65, 0.70, 0.75], 0.03) == False
    print("  PASS  delta exactly at eps → no stop")

    assert plateau_stop([0.70, 0.71], 0.03) == False
    print("  PASS  too short (len=2) → no stop")

    assert plateau_stop([0.30, 0.35, 0.28, 0.33, 0.30], 0.03) == False
    print("  PASS  oscillating → no stop")

    # --- uncertainty filter ---
    print("\n[uncertainty filter]")

    assert not responses_are_substantive(
        ["unknown (no data)", "uncertain (sources vary)",
         "unknown (insufficient)", "unknown (no census)"]
    )
    print("  PASS  all-uncertain → non-substantive (abstain)")

    assert responses_are_substantive(
        ["william shakespeare", "shakespeare",
         "william shakespeare", "shakespeare"]
    )
    print("  PASS  all-factual → substantive (proceed)")

    assert responses_are_substantive(
        ["william shakespeare", "shakespeare",
         "william shakespeare", "unknown"]
    )
    print("  PASS  mostly-factual (3/4) → substantive (proceed)")

    assert not responses_are_substantive(
        ["unknown (no data)", "uncertain",
         "unknown (sources vary)", "william shakespeare"]
    )
    print("  PASS  mostly-uncertain (3/4) → non-substantive (abstain)")

    assert is_uncertainty_response("unknown (no reliable census data)")
    assert is_uncertainty_response("estimates range from 4 to 25 million")
    assert not is_uncertainty_response("william shakespeare")
    assert not is_uncertainty_response("paris")
    print("  PASS  individual is_uncertainty_response checks")

    # --- token_f1 ---
    print("\n[token_f1]")

    assert abs(token_f1("william shakespeare", "shakespeare") - 0.667) < 0.01
    print("  PASS  partial overlap → 0.667")

    assert token_f1("napoleon", "shakespeare") == 0.0
    print("  PASS  no overlap → 0.0")

    # --- check_coverage ---
    print("\n[check_coverage]")

    assert check_coverage([], ["shakespeare"]) == True
    print("  PASS  empty set (abstain) → covered")

    assert check_coverage(["shakespeare"], ["william shakespeare"]) == True
    print("  PASS  partial match F1>=0.5 → covered")

    assert check_coverage(["napoleon"], ["shakespeare"]) == False
    print("  PASS  wrong answer → not covered")

    print("\n" + "=" * 55)
    print("ALL UNIT TESTS PASSED")
    print("=" * 55)

run_all_unit_tests()


UNIT TESTS

[compute_stability]
  PASS  identical responses → 1.0000
  PASS  name variants → 0.9314
  PASS  different people → 0.3941

[plateau_stop]
  PASS  flat history → stop
  PASS  rising history → no stop
  PASS  delta exactly at eps → no stop
  PASS  too short (len=2) → no stop
  PASS  oscillating → no stop

[uncertainty filter]
  PASS  all-uncertain → non-substantive (abstain)
  PASS  all-factual → substantive (proceed)
  PASS  mostly-factual (3/4) → substantive (proceed)
  PASS  mostly-uncertain (3/4) → non-substantive (abstain)
  PASS  individual is_uncertainty_response checks

[token_f1]
  PASS  partial overlap → 0.667
  PASS  no overlap → 0.0

[check_coverage]
  PASS  empty set (abstain) → covered
  PASS  partial match F1>=0.5 → covered
  PASS  wrong answer → not covered

ALL UNIT TESTS PASSED


In [ ]:
DEFAULT_CONFIG = SamplerConfig()

In [ ]:
def run_integration_test():
    cfg = DEFAULT_CONFIG

    test_cases = [
        (
            "Who wrote the play Hamlet?",
            ["shakespeare", "william shakespeare"],
            "EASY",
            "expect: abstain=False, reason=plateau, N~0.04, covered=True",
        ),
        (
            "What caused the fall of the Roman Empire?",
            ["decline", "barbarian invasions", "multiple factors"],
            "MEDIUM",
            "expect: abstain=True, reason=budget_exhausted",
        ),
        (
            "What was the exact population of the Byzantine Empire in 500 AD?",
            ["unknown"],
            "HARD",
            "expect: abstain=True, reason=uncertainty_responses",
        ),
    ]

    print(f"\nINTEGRATION TEST\n{'='*65}")

    for question, gold, label, expectation in test_cases:
        clear_embed_cache()
        result = adaptive_sample(question, config=cfg)
        covered = check_coverage(
            result["cleaned"] if not result["abstain"] else [],
            gold,
        )

        print(f"\n[{label}] {expectation}")
        print(f"  Q        : {question}")
        print(f"  abstain  : {result['abstain']}")
        print(f"  reason   : {result['reason']}")
        print(f"  n_score  : {result['n_score']}")
        print(f"  batches  : {result['n_batches']}/{cfg.max_batches}")
        print(f"  stability: {result['stability']:.3f}")
        print(f"  history  : {[round(h, 3) for h in result['history']]}")
        print(f"  responses: {list(set(result['cleaned']))[:5]}")
        print(f"  covered  : {covered}")

run_integration_test()



INTEGRATION TEST

[EASY] expect: abstain=False, reason=plateau, N~0.04, covered=True
  Q        : Who wrote the play Hamlet?
  abstain  : False
  reason   : plateau
  n_score  : 0.057151
  batches  : 3/6
  stability: 0.943
  history  : [0.931, 0.938, 0.943]
  responses: ['shakespeare', 'william shakespeare']
  covered  : True

[MEDIUM] expect: abstain=True, reason=budget_exhausted
  Q        : What caused the fall of the Roman Empire?
  abstain  : False
  reason   : plateau
  n_score  : 0.353954
  batches  : 6/6
  stability: 0.646
  history  : [0.857, 0.797, 0.744, 0.63, 0.633, 0.646]
  responses: ['decline & migration (multi-factorial)', 'decline/internal factors', 'decline & migration (bbce)', 'decline & migration (multiple factors)', 'decline & migration (various factors)']
  covered  : True

[HARD] expect: abstain=True, reason=uncertainty_responses
  Q        : What was the exact population of the Byzantine Empire in 500 AD?
  abstain  : True
  reason   : uncertainty_responses
  n

In [ ]:
def test_compute_qhat():
    print("Testing compute_qhat...\n")

    # Test 1: worked example from earlier explanation
    # n=5, alpha=0.1 → position = ceil(6 * 0.9) = ceil(5.4) = 6 > 5 → inf
    scores = [0.01, 0.03, 0.29, 0.42, 0.69]
    q = compute_qhat(scores, alpha=0.1)
    assert q == float("inf"), f"Expected inf for n=5, got {q}"
    print(f"  PASS  n=5, alpha=0.1 → q_hat=inf (position > n)")

    # Test 2: n=9, alpha=0.1 → position = ceil(10 * 0.9) = ceil(9.0) = 9
    # sorted[8] = max score
    scores = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
    q = compute_qhat(scores, alpha=0.1)
    assert q == 0.70, f"Expected 0.70, got {q}"
    print(f"  PASS  n=9, alpha=0.1 → q_hat={q:.2f} (9th of 9)")

    # Test 3: n=100, alpha=0.1 → position = ceil(101 * 0.9) = ceil(90.9) = 91
    import random
    random.seed(42)
    scores = sorted([random.random() for _ in range(100)])
    q = compute_qhat(scores, alpha=0.1)
    assert q == scores[90], f"Expected scores[90], got {q}"
    print(f"  PASS  n=100, alpha=0.1 → q_hat=scores[90]={q:.4f}")

    # Test 4: n=500, alpha=0.1 → position = ceil(501 * 0.9) = ceil(450.9) = 451
    scores_500 = sorted([random.random() for _ in range(500)])
    q = compute_qhat(scores_500, alpha=0.1)
    assert q == scores_500[450], f"Expected scores_500[450]"
    print(f"  PASS  n=500, alpha=0.1 → q_hat=scores[450]={q:.4f}")

    # Test 5: alpha=0.2 → position = ceil(501 * 0.8) = ceil(400.8) = 401
    q_lenient = compute_qhat(scores_500, alpha=0.2)
    assert q_lenient <= q, "More lenient alpha should give smaller or equal q_hat"
    print(f"  PASS  alpha=0.2 gives smaller q_hat ({q_lenient:.4f} <= {q:.4f})")

    print("\nAll compute_qhat tests passed.\n")

test_compute_qhat()

Testing compute_qhat...

  NOTE: position=6 > n=5. q_hat = inf (conservative).
  PASS  n=5, alpha=0.1 → q_hat=inf (position > n)
  PASS  n=9, alpha=0.1 → q_hat=0.70 (9th of 9)
  PASS  n=100, alpha=0.1 → q_hat=scores[90]=0.8978
  PASS  n=500, alpha=0.1 → q_hat=scores[450]=0.9012
  PASS  alpha=0.2 gives smaller q_hat (0.8171 <= 0.9012)

All compute_qhat tests passed.



In [ ]:
ALPHA = 0.1

# tesstttt


In [ ]:
LOFREEFCP_TRIVIAQA = {
    0.20: {"ECR": 80.1, "SSC": 79.0, "APSS": 2.19},
    0.25: {"ECR": 75.3, "SSC": 74.5, "APSS": 1.43},
    0.30: {"ECR": 70.3, "SSC": 76.7, "APSS": 1.08},
    0.35: {"ECR": 65.1, "SSC": 78.5, "APSS": 0.90},
    0.40: {"ECR": 60.0, "SSC": 81.0, "APSS": 0.75},
    0.45: {"ECR": 55.2, "SSC": 84.1, "APSS": 0.66},
}
LOFREEFCP_WEBQ = {
    0.35: {"ECR": 65.1, "SSC": 61.1, "APSS": 5.33},
    0.40: {"ECR": 60.0, "SSC": 60.0, "APSS": 2.68},
    0.45: {"ECR": 55.1, "SSC": 60.1, "APSS": 1.60},
    0.50: {"ECR": 50.3, "SSC": 59.9, "APSS": 1.06},
}

In [ ]:
TRIVIAQA_ALPHAS = [0.25, 0.30, 0.35, 0.40, 0.45]

triviaqa_results = run_full_evaluation(
    DATASETS,
    dataset_name = "triviaqa",
    alphas       = TRIVIAQA_ALPHAS,
    lofree_ref   = LOFREEFCP_TRIVIAQA,
    cal_samples  = 500,
    test_samples = 500,
)

print_comparison_table(triviaqa_results, LOFREEFCP_TRIVIAQA, "TriviaQA")

# WebQ — match LofreeCP's alpha range
WEBQ_ALPHAS = [0.35, 0.40, 0.45, 0.50]

webq_results = run_full_evaluation(
    DATASETS,
    dataset_name = "webq",
    alphas       = WEBQ_ALPHAS,
    lofree_ref   = LOFREEFCP_WEBQ,
    cal_samples  = 500,
    test_samples = 500,
)

print_comparison_table(webq_results, LOFREEFCP_WEBQ, "WebQuestions")

# Save all results
import json
with open("/content/final_results.json", "w") as f:
    # Strip full result lists for storage
    save = []
    for r in triviaqa_results + webq_results:
        d = {k: v for k, v in r.items() if k != "results"}
        save.append(d)
    json.dump(save, f, indent=2)
print("\nSaved to /content/final_results.json")



FULL EVALUATION: TRIVIAQA

--- alpha=0.25 (target ECR >= 75%) ---

Calibrating on triviaqa (n=500, alpha=0.25)
Config: SamplerConfig(batch_size=3, max_batches=6, eps=0.03, temperature=0.9, min_batches=2)
-------------------------------------------------------
  [  50/500] abstain_rate=10.0%  coverage=86.0%  eta=63.6min
  [ 100/500] abstain_rate=8.0%  coverage=84.0%  eta=54.4min
  [ 150/500] abstain_rate=8.7%  coverage=84.0%  eta=45.6min
  [ 200/500] abstain_rate=9.5%  coverage=84.5%  eta=37.5min
  [ 250/500] abstain_rate=10.4%  coverage=85.2%  eta=31.1min
  [ 300/500] abstain_rate=9.7%  coverage=85.0%  eta=24.3min
  [ 350/500] abstain_rate=10.0%  coverage=84.9%  eta=18.4min
  [ 400/500] abstain_rate=10.0%  coverage=83.2%  eta=12.3min
  [ 450/500] abstain_rate=10.7%  coverage=82.9%  eta=6.2min
  [ 500/500] abstain_rate=11.4%  coverage=82.6%  eta=0.0min

Calibration complete — triviaqa
  n_scored     : 443
  n_abstained  : 57 (11.4%)
  q_hat        : 0.1389
  cal coverage : 82.6%  (diag